In [2]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

#setup project root and paths
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set NIJ root
nij_root = project_root / "data" / "processed"

#path to put results
output_dir = project_root/"results"/"NIJ"/"graphs_DAGSLAM"
output_dir.mkdir(parents=True,exist_ok=True)

#test it works
print("nij root:" , nij_root)
print("Output directory:", output_dir)

nij root: /dcs/23/u2200504/thesis/recidivism-causal/data/processed
Output directory: /dcs/23/u2200504/thesis/recidivism-causal/results/NIJ/graphs_DAGSLAM


In [3]:
#add dagslam implementation to system path
dagslam_path = project_root/"code"/"dagslam"/"DAGSLAM Causal Bayesian Network Structure Learning of Mixed Type Data"/"_dagslam"
sys.path.append(str(dagslam_path))

#uses DAGSLAM implementation code authored by Yuanyuan Zhao. 
#https://github.com/yuanyuan-zhao-pku/DAGSLAM/blob/main/DAGSLAM%20Causal%20Bayesian%20Network%20Structure%20Learning%20of%20Mixed%20Type%20Data/_dagslam/DAGSLAM.py
#DAGSLAM is an extension of the NOTEARS algorithm developed by Xun Zheng, et al.
import importlib
import DAGSLAM 
importlib.reload(DAGSLAM)

from DAGSLAM import dagslam

#m_vec gives the total number of categories for each multinomial variable
#helper function to infer loss types and generate m_vec for each column
def infer_type(df):
    loss_type=[]
    m_vec=[]

    for col in df.columns:
        x=df[col].dropna()
        unique_vals=x.unique()
        num_unique = len(unique_vals)

        #infer variable type by combination of dtype and cardinality
        if np.issubdtype(x.dtype, np.number):
            #numeric datatypes
            if num_unique == 2 and set(unique_vals).issubset({0,1}): #binary numeric variable
                loss_type.append("logistic")
                m_vec.append(1) # 1 "category"
            else: #continuous 
                loss_type.append("gauss")
                m_vec.append(1)
        else: #non-numeric
            if num_unique == 2:
                #binary categorical 
                loss_type.append("logistic")
                m_vec.append(1)
            else:
                #multi-class categorical
                loss_type.append("multi-logistic")
                m_vec.append(num_unique)
    print("loss_type:", loss_type)
    print("m_vec:", m_vec)
    print("n_cols:", len(df.columns), "len(loss_type):", len(loss_type))
    
    return loss_type, m_vec

def encode_mixed_df(df):
    df_enc = df.copy()
    for col in df_enc.columns:
        #use categorical code for non numeric
        if not np.issubdtype(df_enc[col].dtype, np.number):
            df_enc[col] = df_enc[col].astype("category").cat.codes
    return df_enc


def run_dagslam(df, lambda1=0.03, max_iter=100, w_threshold=0.25): #changed lambda 1 0.03 ->0.1 w_threshold 0.25->0.3
    df_clean = df.dropna().copy()
    df_enc = encode_mixed_df(df_clean)
    
    loss_type, m_vec =infer_type(df_clean)
    assert list(df_clean.columns) == list(df_enc.columns)
    X=df_enc.to_numpy(dtype=float)

    start = time.time() 
    W_est = dagslam(X, loss_type=loss_type, m_vec=m_vec, lambda1=lambda1,max_iter=max_iter, w_threshold=w_threshold)
    end = time.time()
    print(f"DAGSLAM took {(end - start)/60:.2f} minutes")

    return W_est

#draw graphs based on DAGSLAM weighted adjacency matrix
def draw_graph(W, output_path, node_labels=None,threshold=0):
    n = W.shape[0] # number of nodes
    G = nx.DiGraph() # initialise empty directed graph
    
    #initialise default node labels
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]

    #add nodes to digraph
    for i, name in enumerate(node_labels):
        G.add_node(i, label=name)

    #add edges for surviving weights (thresholding done during the dagslam phase)
    for i in range(n):
        for j in range(n): # for each possible edge 
            w=W[i,j]
            if abs(w)>threshold:
                G.add_edge(i, j, weight=w)

    plt.figure(figsize=(6,6))
    pos = nx.spring_layout(G, seed=0)
    nx.draw(G, pos, with_labels=True, labels={i: node_labels[i] for i in range(n)},
            node_size=800, font_size=8, arrowsize=10)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

In [4]:
from graphviz import Digraph
from pathlib import Path
import numpy as np

def draw_graphviz_dag(W, out_path, node_labels=None, threshold=0.0, engine="dot"):

    W = np.asarray(W)
    if W.ndim == 1:
        W = W.reshape(1, 1)

    n = W.shape[0]

    # Default labels use X0, X1,...
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]
    else:
        node_labels = list(node_labels)[:n]

    out_path = Path(out_path)
    g = Digraph(format="svg", engine=engine)

    g.attr(rankdir="LR")
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10",
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7",
    )

    # Add nodes
    for name in node_labels:
        g.node(name, label=name)

    # Add edges for weights above threshold
    for i, src in enumerate(node_labels):
        for j, tgt in enumerate(node_labels):
            w = W[i, j]
            if abs(w) > threshold:
                # You can optionally add weight as label=str(round(w, 2))
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)


In [5]:
import time
csv_path = nij_root/"NIJ_lean_compact_onehot.csv"
df = pd.read_csv(csv_path)

# Run DAGSLAM on this dataset
W_est = run_dagslam(df)

#save recovered adjacency matrix to a csv
adj_df = pd.DataFrame(W_est, index=df.columns, columns=df.columns)
adj_out_path = output_dir / "NIJ_graph_DAGSLAM_adj.csv"
adj_df.to_csv(adj_out_path, index=True)

out_path = output_dir / "NIJ_graph_DAGSLAM"

#use graphviz instead
draw_graphviz_dag(W_est, out_path, node_labels=list(df.columns), threshold=0.0)

loss_type: ['logistic', 'logistic', 'logistic', 'logistic', 'logistic', 'logistic', 'logistic', 'logistic', 'logistic', 'logistic', 'logistic', 'logistic', 'logistic', 'logistic', 'gauss', 'gauss', 'gauss', 'gauss', 'logistic', 'gauss', 'logistic']
m_vec: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
n_cols: 21 len(loss_type): 21
iter:0
rho:1.0
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=9803.13604777547
loss=6834.360036315092
loss=6834.300297068414
loss=6834.081795152393
loss=6832.074701733789
loss=6826.45455

In [16]:
import networkx as nx
from collections import Counter

W_est = np.asarray(W_est)
if W_est.ndim == 1:
    W_est = W_est.reshape(1, 1)

n = W_est.shape[0] # number of nodes
G = nx.DiGraph() # initialise empty directed graph
node_labels=list(df.columns)

#add nodes to digraph
for i, name in enumerate(node_labels):
    G.add_node(i, label=name)

#add edges for surviving weights (thresholding done during the dagslam phase)
for i in range(n):
    for j in range(n): # for each possible edge 
        w=W_est[i,j]
        if abs(w)>0:
            G.add_edge(i, j, weight=w)

AGE_PREFIX = "Age_at_Release_"          # prefix of one-hot age bucket columns
AGE_MERGED = "Age_at_Release"           # name of merged age node
TARGET     = "Recidivism_Within_3years" #target node
#G assumed to be an nx.DiGraph()

#Merge age buckets
age_nodes = [n for n in G.nodes if isinstance(n, str) and n.startswith(AGE_PREFIX)]
G_merged = nx.DiGraph()

# copy all nodes except age buckets
for n in G.nodes:
    if n not in age_nodes:
        G_merged.add_node(n)

# add merged age node if any buckets exist
if age_nodes:
    G_merged.add_node(AGE_MERGED)

age_out = Counter()   #counts outgoing edges from age buckets Age -> X (bucket -> non-age)
age_in  = Counter()   #counts incoming edges ".. "(non-age -> bucket)

for u, v, data in G.edges(data=True):

    #ignore edges between age buckets
    if u in age_nodes and v in age_nodes:
        continue

    #edges not affecting age buckets are just copied over
    if u not in age_nodes and v not in age_nodes:
        G_merged.add_edge(u, v, **data)
        continue

    #bucket -> non-age variable
    if u in age_nodes and v not in age_nodes:
        age_out[v] += 1

    #non-age variable -> bucket
    if v in age_nodes and u not in age_nodes:
        age_in[u] += 1

#decide majority direction for each neighbour of an age bucket
for x, cnt_out in age_out.items():
    cnt_in = age_in.get(x, 0)

    if cnt_out > cnt_in:
        # majority oriented as Age bucket -> X
        G_merged.add_edge(AGE_MERGED, x)
    elif cnt_in > cnt_out:
        # majority X -> Age bucket
        G_merged.add_edge(x, AGE_MERGED)
    else:
        #tie: drop, dont add edge to age merged
        pass

# add edges X -> Age for neighbours that only ever had incoming-to-bucket edges
for x, cnt_in in age_in.items():
    if x in age_out:
        continue    # already handled above
    G_merged.add_edge(x, AGE_MERGED)

if TARGET not in G_merged:
    raise ValueError(f"Target node '{TARGET}' not found in graph; "
                     f"available nodes include: {list(G_merged.nodes)[:10]} ...")

parents  = set(G_merged.predecessors(TARGET))
markov_blanket_nodes = parents | {TARGET}

G_mb = G_merged.subgraph(markov_blanket_nodes).copy()

print("Original nodes:", len(G.nodes))
print("After age-merge:", len(G_merged.nodes))
print("Markov blanket nodes:", len(G_mb.nodes))

node_labels = list(G_mb.nodes())          
adj = nx.to_numpy_array(G_mb, nodelist=node_labels, dtype=int)

out_path = output_dir/"NIJ_graph_DAGSLAM_PRUNED"
draw_graphviz_dag(adj, out_path, node_labels=node_labels, engine="dot")

ValueError: Target node 'Recidivism_Within_3years' not found in graph; available nodes include: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] ...

In [17]:
len(node_labels)

21

In [18]:
print(n)

20
